# Exploratory Data Analysis - UCI Individual Household Electric Power Consumption

**TFG:** Detection of Electricity Consumption Anomalies in Edge Computing for the Mitigation of FDI Attacks

**Author:** Matvey Sinelnik

**Goals of this notebook:**
1. Load and clean the UCI Household dataset (train + test)
2. Carry out a full exploratory analysis (distributions, temporal patterns, correlations)
3. Characterise "normal" consumption behaviour, which serves as the baseline for the FDI attacks injected later
4. Document the findings for Chapter 4 of the thesis

## 0. Setup and Installation

In [ ]:
# Install dependencies if needed
!pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12

print("Libraries loaded successfully")

## 1. Data Loading

Adjust the paths to match your repository layout. The expected structure is:
```
data/
  ├── train.csv
  └── test.csv
```

In [ ]:
# ============================================================
# ADJUST THESE PATHS to match your folder structure
# ============================================================
TRAIN_PATH = '/Volumes/Extreme Pro Particion 1TB/TFG/UCIrvine/household_power_consumption_train.csv'
TEST_PATH = '/Volumes/Extreme Pro Particion 1TB/TFG/UCIrvine/household_power_consumption_test.csv'

# Load train and test
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

print(f"Train: {df_train.shape[0]:,} rows, {df_train.shape[1]} columns")
print(f"Test:  {df_test.shape[0]:,} rows, {df_test.shape[1]} columns")
print(f"Total: {df_train.shape[0] + df_test.shape[0]:,} rows")
print(f"\nColumns: {list(df_train.columns)}")

In [ ]:
# Data preview
print("=" * 60)
print("TRAIN - First rows")
print("=" * 60)
display(df_train.head(10))

print("\n" + "=" * 60)
print("TEST - First rows")
print("=" * 60)
display(df_test.head(10))

In [ ]:
# Data type info
print("Data types (train):")
print(df_train.dtypes)
print(f"\nTrain memory: {df_train.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Test memory:  {df_test.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 2. Cleaning and Preprocessing

The original dataset encodes missing values as `?`. Depending on how the train/test split was
produced, these may or may not already be handled, so we verify and clean them here.

In [ ]:
def clean_dataset(df, name='dataset'):
    """
    Clean the UCI Household dataset end to end:
    - Replace '?' with NaN
    - Build a datetime index
    - Coerce numeric columns
    - Compute Sub_metering_rest (non-submetered consumption)
    """
    df = df.copy()

    # Replace the '?' missing-value marker with NaN
    df.replace('?', np.nan, inplace=True)

    # Build the datetime column, trying the formats the CSVs may use
    if 'Date' in df.columns and 'Time' in df.columns:
        # Original format: dd/mm/yyyy + hh:mm:ss
        df['datetime'] = pd.to_datetime(
            df['Date'] + ' ' + df['Time'],
            format='%d/%m/%Y %H:%M:%S',
            errors='coerce'
        )
    elif 'datetime' in df.columns or 'Datetime' in df.columns:
        col = 'datetime' if 'datetime' in df.columns else 'Datetime'
        df['datetime'] = pd.to_datetime(df[col], errors='coerce')
    else:
        # Unexpected schema: fall back to the first two columns as Date + Time
        print(f"  [WARNING] Detected columns: {list(df.columns)}")
        print(f"  Falling back to the first two columns as Date + Time...")
        df['datetime'] = pd.to_datetime(
            df.iloc[:, 0].astype(str) + ' ' + df.iloc[:, 1].astype(str),
            errors='coerce'
        )

    # Numeric columns
    numeric_cols = [
        'Global_active_power', 'Global_reactive_power',
        'Voltage', 'Global_intensity',
        'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]

    # Keep only the columns that are actually present (names may differ slightly)
    existing_numeric = [c for c in numeric_cols if c in df.columns]

    for col in existing_numeric:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Derive Sub_metering_rest: active power not captured by the three sub-meters
    if all(c in df.columns for c in ['Global_active_power', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']):
        df['Sub_metering_rest'] = (
            df['Global_active_power'] * 1000 / 60
            - df['Sub_metering_1']
            - df['Sub_metering_2']
            - df['Sub_metering_3']
        )

    # Use datetime as a sorted index
    if df['datetime'].notna().sum() > 0:
        df = df.set_index('datetime')
        df = df.sort_index()

    # Drop the now-redundant Date/Time columns
    for col in ['Date', 'Time']:
        if col in df.columns:
            df.drop(col, axis=1, inplace=True)

    # Summary
    n_missing = df.isna().sum().sum()
    pct_missing = (n_missing / (df.shape[0] * df.shape[1])) * 100
    print(f"\n[{name}] Cleaning done:")
    print(f"  Rows: {df.shape[0]:,}")
    print(f"  Time range: {df.index.min()} -> {df.index.max()}")
    print(f"  Missing values: {n_missing:,} ({pct_missing:.2f}%)")

    return df


# Clean both splits
df_train = clean_dataset(df_train, 'TRAIN')
df_test = clean_dataset(df_test, 'TEST')

In [ ]:
# Merge both splits for the global exploratory analysis
# (the train/test separation is preserved for the modelling stage)
df_full = pd.concat([df_train, df_test]).sort_index()
df_full = df_full[~df_full.index.duplicated(keep='first')]  # drop any duplicate timestamps

print(f"Full dataset: {df_full.shape[0]:,} rows")
print(f"Range: {df_full.index.min()} -> {df_full.index.max()}")
print(f"Duration: {(df_full.index.max() - df_full.index.min()).days} days")

### 2.1 Missing Value Analysis

In [ ]:
# Missing values per column
missing = df_full.isna().sum()
missing_pct = (missing / len(df_full)) * 100

missing_df = pd.DataFrame({
    'Missing values': missing,
    'Percentage (%)': missing_pct.round(2)
})
missing_df = missing_df[missing_df['Missing values'] > 0]

print("Missing values per column:")
display(missing_df)

# Visualise how missing values are distributed over time
fig, ax = plt.subplots(figsize=(14, 3))
missing_by_day = df_full['Global_active_power'].isna().resample('D').sum()
ax.bar(missing_by_day.index, missing_by_day.values, width=1, color='crimson', alpha=0.7)
ax.set_title('Missing Values per Day (Global Active Power)', fontsize=14)
ax.set_ylabel('Missing minutes')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

print(f"\nDays fully missing (1440 min): "
      f"{(missing_by_day == 1440).sum()}")
print(f"Days with any missing value: {(missing_by_day > 0).sum()}")

In [ ]:
# Imputation strategy: linear interpolation for short gaps (<= 10 min),
# leave longer gaps as NaN (they get dropped when building windows)

numeric_cols = df_full.select_dtypes(include=[np.number]).columns.tolist()

df_clean = df_full.copy()
df_clean[numeric_cols] = df_clean[numeric_cols].interpolate(
    method='linear', limit=10  # interpolate gaps of at most 10 minutes
)

remaining_missing = df_clean[numeric_cols].isna().sum().sum()
print(f"Missing values before interpolation: {df_full[numeric_cols].isna().sum().sum():,}")
print(f"Missing values after interpolation (limit=10): {remaining_missing:,}")

# Drop the rows that still contain NaN
df_clean = df_clean.dropna(subset=numeric_cols)
print(f"Final rows after dropping remaining NaN: {df_clean.shape[0]:,}")
print(f"Data retained: {df_clean.shape[0]/df_full.shape[0]*100:.1f}%")

## 3. Descriptive Statistics

In [ ]:
# Full descriptive statistics
stats = df_clean[numeric_cols].describe().T
stats['missing_pct'] = ((df_full[numeric_cols].isna().sum() / len(df_full)) * 100).round(2)
stats['skew'] = df_clean[numeric_cols].skew().round(3)
stats['kurtosis'] = df_clean[numeric_cols].kurtosis().round(3)

print("Full descriptive statistics:")
display(stats.round(4))

In [ ]:
# Distribution of each variable
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

plot_cols = [c for c in numeric_cols if c != 'Sub_metering_rest'] + ['Sub_metering_rest']
plot_cols = [c for c in plot_cols if c in df_clean.columns][:8]

for i, col in enumerate(plot_cols):
    ax = axes[i]
    data = df_clean[col].dropna()

    ax.hist(data, bins=100, color='steelblue', alpha=0.7, edgecolor='none')
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1, label=f'mean={data.mean():.2f}')
    ax.axvline(data.median(), color='orange', linestyle='--', linewidth=1, label=f'median={data.median():.2f}')
    ax.legend(fontsize=8)
    ax.set_ylabel('Frequency')

# Hide unused axes
for j in range(len(plot_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribution of each variable', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Temporal Patterns

Characterising the normal consumption patterns is **essential** for detecting anomalies later on:
an FDI attack becomes detectable precisely when it breaks these patterns.

### 4.1 Daily Pattern (average profile by hour of day)

In [ ]:
# Average consumption profile by hour of day
df_clean['hour'] = df_clean.index.hour
df_clean['dayofweek'] = df_clean.index.dayofweek  # 0=Monday, 6=Sunday
df_clean['month'] = df_clean.index.month
df_clean['year'] = df_clean.index.year
df_clean['is_weekend'] = df_clean['dayofweek'].isin([5, 6]).astype(int)

hourly_profile = df_clean.groupby('hour')['Global_active_power'].agg(['mean', 'std', 'median'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hourly_profile.index, hourly_profile['mean'], 'b-o', linewidth=2, label='Mean')
ax.fill_between(
    hourly_profile.index,
    hourly_profile['mean'] - hourly_profile['std'],
    hourly_profile['mean'] + hourly_profile['std'],
    alpha=0.2, color='blue', label='+/- 1 std. dev.'
)
ax.plot(hourly_profile.index, hourly_profile['median'], 'r--', linewidth=1.5, label='Median')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Global Active Power (kW)')
ax.set_title('Average Consumption Profile by Hour of Day', fontsize=14, fontweight='bold')
ax.set_xticks(range(24))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nObservations:")
print(f"  Lowest-consumption hour: {hourly_profile['mean'].idxmin()}:00 ({hourly_profile['mean'].min():.3f} kW)")
print(f"  Highest-consumption hour: {hourly_profile['mean'].idxmax()}:00 ({hourly_profile['mean'].max():.3f} kW)")
print(f"  Max/min ratio: {hourly_profile['mean'].max() / hourly_profile['mean'].min():.1f}x")

### 4.2 Weekday vs Weekend

In [ ]:
# Weekday vs weekend profile
weekday_profile = df_clean[df_clean['is_weekend'] == 0].groupby('hour')['Global_active_power'].mean()
weekend_profile = df_clean[df_clean['is_weekend'] == 1].groupby('hour')['Global_active_power'].mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(weekday_profile.index, weekday_profile.values, 'b-o', linewidth=2, markersize=4, label='Weekday (Mon-Fri)')
ax.plot(weekend_profile.index, weekend_profile.values, 'r-s', linewidth=2, markersize=4, label='Weekend (Sat-Sun)')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Global Active Power (kW)')
ax.set_title('Consumption Profile: Weekday vs Weekend', fontsize=14, fontweight='bold')
ax.set_xticks(range(24))
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3 Weekly Pattern

In [ ]:
# Average consumption by day of the week
days_en = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly_profile = df_clean.groupby('dayofweek')['Global_active_power'].agg(['mean', 'std'])

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(7), weekly_profile['mean'], yerr=weekly_profile['std'],
              color=['steelblue']*5 + ['coral']*2, alpha=0.8, capsize=5)
ax.set_xticks(range(7))
ax.set_xticklabels(days_en)
ax.set_ylabel('Global Active Power (kW)')
ax.set_title('Average Consumption by Day of the Week', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### 4.4 Seasonal Pattern (monthly)

In [ ]:
# Average consumption by month
months_en = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
             'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

monthly_profile = df_clean.groupby('month')['Global_active_power'].agg(['mean', 'std'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(1, 13), monthly_profile['mean'], yerr=monthly_profile['std'],
       color='steelblue', alpha=0.8, capsize=5)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months_en)
ax.set_ylabel('Global Active Power (kW)')
ax.set_title('Average Consumption by Month (seasonal pattern)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Highest-consumption month: {months_en[monthly_profile['mean'].idxmax()-1]} ({monthly_profile['mean'].max():.3f} kW)")
print(f"Lowest-consumption month: {months_en[monthly_profile['mean'].idxmin()-1]} ({monthly_profile['mean'].min():.3f} kW)")

### 4.5 Full Time Series (overview)

In [ ]:
# Resample to daily resolution for plotting
daily = df_clean['Global_active_power'].resample('D').mean()

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Full daily series
ax = axes[0]
ax.plot(daily.index, daily.values, linewidth=0.8, color='steelblue', alpha=0.8)
ax.axhline(daily.mean(), color='red', linestyle='--', linewidth=1, alpha=0.7,
           label=f'Overall mean: {daily.mean():.3f} kW')
ax.set_title('Average Daily Consumption - Full Series', fontsize=14, fontweight='bold')
ax.set_ylabel('Global Active Power (kW)')
ax.legend()
ax.grid(True, alpha=0.3)

# Moving averages to expose the underlying trend
ax = axes[1]
rolling_7d = daily.rolling(window=7, center=True).mean()
rolling_30d = daily.rolling(window=30, center=True).mean()
ax.plot(daily.index, daily.values, linewidth=0.5, color='lightblue', alpha=0.5, label='Daily')
ax.plot(rolling_7d.index, rolling_7d.values, linewidth=1.5, color='steelblue', label='7-day moving average')
ax.plot(rolling_30d.index, rolling_30d.values, linewidth=2, color='darkred', label='30-day moving average')
ax.set_title('Consumption Trend (moving averages)', fontsize=14, fontweight='bold')
ax.set_ylabel('Global Active Power (kW)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.6 Heatmap: Consumption by hour and day of the week

In [ ]:
# Heatmap of consumption: hour vs day of the week
pivot = df_clean.groupby(['dayofweek', 'hour'])['Global_active_power'].mean().unstack()

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    pivot, cmap='YlOrRd', annot=False, fmt='.2f',
    xticklabels=[f'{h}:00' for h in range(24)],
    yticklabels=days_en,
    ax=ax, cbar_kws={'label': 'kW'}
)
ax.set_title('Heatmap: Average Consumption by Hour and Day', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of day')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 5. Sub-Metering Analysis

The three sub-meters plus the derived "rest" channel reveal how the total consumption is composed.

In [ ]:
# Contribution of each sub-meter to total consumption
sub_cols = ['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
if 'Sub_metering_rest' in df_clean.columns:
    sub_cols.append('Sub_metering_rest')

sub_means = df_clean[sub_cols].mean()
sub_labels = {
    'Sub_metering_1': 'Kitchen\n(dishwasher, oven, microwave)',
    'Sub_metering_2': 'Laundry\n(washing machine, dryer, refrigerator)',
    'Sub_metering_3': 'Water heater\n+ Air conditioning',
    'Sub_metering_rest': 'Rest\n(non-submetered)'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
labels = [sub_labels.get(c, c) for c in sub_cols]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
axes[0].pie(sub_means.values, labels=labels, autopct='%1.1f%%',
            colors=colors[:len(sub_cols)], startangle=90)
axes[0].set_title('Consumption Breakdown by Zone', fontsize=13, fontweight='bold')

# Hourly profile per sub-meter
for col in sub_cols:
    hourly = df_clean.groupby('hour')[col].mean()
    axes[1].plot(hourly.index, hourly.values, '-o', markersize=3,
                linewidth=1.5, label=sub_labels.get(col, col).replace('\n', ' '))

axes[1].set_xlabel('Hour of day')
axes[1].set_ylabel('Consumption (Wh)')
axes[1].set_title('Hourly Profile by Sub-Metering Zone', fontsize=13, fontweight='bold')
axes[1].set_xticks(range(24))
axes[1].legend(fontsize=8, loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Correlations between Variables

In [ ]:
# Correlation matrix
corr_cols = [c for c in numeric_cols if c in df_clean.columns]
if 'Sub_metering_rest' in df_clean.columns:
    corr_cols.append('Sub_metering_rest')

corr = df_clean[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Report the strong correlations
print("\nStrongest correlations (|r| > 0.5):")
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.5:
            print(f"  {corr.columns[i]} <-> {corr.columns[j]}: r = {r:.3f}")

## 7. Preliminary Anomaly Detection (before injecting attacks)

Before injecting any FDI attacks, we check whether the dataset already contains natural anomalies.
This helps us calibrate the boundary between "rare but legitimate" behaviour and a genuine attack.

In [ ]:
# Simple baseline: per-hour Z-score on Global_active_power
# (a spike at 3am is not comparable to one at 7pm, so we normalise by hour)

hourly_stats = df_clean.groupby('hour')['Global_active_power'].agg(['mean', 'std'])

df_clean['gap_zscore'] = df_clean.apply(
    lambda row: (row['Global_active_power'] - hourly_stats.loc[row['hour'], 'mean'])
                / hourly_stats.loc[row['hour'], 'std']
    if hourly_stats.loc[row['hour'], 'std'] > 0 else 0,
    axis=1
)

# Flag points beyond |z| > 3
threshold = 3
anomalies = df_clean[abs(df_clean['gap_zscore']) > threshold]

print(f"Natural anomalies detected (|z| > {threshold}): {len(anomalies):,}")
print(f"Percentage: {len(anomalies)/len(df_clean)*100:.2f}%")

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Z-score distribution
ax = axes[0]
ax.hist(df_clean['gap_zscore'], bins=200, color='steelblue', alpha=0.7, edgecolor='none')
ax.axvline(-threshold, color='red', linestyle='--', label=f'Threshold +/- {threshold}')
ax.axvline(threshold, color='red', linestyle='--')
ax.set_title('Distribution of Z-Scores (normalised by hour)', fontsize=14, fontweight='bold')
ax.set_xlabel('Z-Score')
ax.set_ylabel('Frequency')
ax.set_xlim(-6, 6)
ax.legend()

# Anomalies over time
ax = axes[1]
daily_power = df_clean['Global_active_power'].resample('D').mean()
daily_anomaly_count = anomalies['Global_active_power'].resample('D').count()
ax.plot(daily_power.index, daily_power.values, linewidth=0.5, color='steelblue', alpha=0.5)
ax.scatter(daily_anomaly_count.index, 
           daily_power.reindex(daily_anomaly_count.index).values,
           c='red', s=daily_anomaly_count.values, alpha=0.3,
           label='Days with anomalies (size = count)')
ax.set_title('When the Natural Anomalies Occur', fontsize=14, fontweight='bold')
ax.set_ylabel('Global Active Power (kW)')
ax.legend()

plt.tight_layout()
plt.show()

## 8. Stationarity and Autocorrelation Analysis

These results inform the choice of model (LSTM vs Autoencoder vs Isolation Forest).

In [21]:
!pip install statsmodels

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ADF test on hourly data (minute resolution would be too heavy)
hourly_gap = df_clean['Global_active_power'].resample('h').mean().dropna()

result = adfuller(hourly_gap, maxlag=168)  # max lag = 1 week in hours
print("Augmented Dickey-Fuller test (hourly data):")
print(f"  ADF statistic: {result[0]:.4f}")
print(f"  p-value: {result[1]:.6f}")
print(f"  Lags used: {result[2]}")
print(f"  Critical values:")
for key, val in result[4].items():
    print(f"    {key}: {val:.4f}")
print(f"\n  -> Series is {'STATIONARY' if result[1] < 0.05 else 'NON-STATIONARY'} (p < 0.05)")

In [ ]:
# Autocorrelation (ACF and PACF)
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# ACF up to 168 lags (= 1 week in hours)
plot_acf(hourly_gap, lags=168, ax=axes[0], alpha=0.05)
axes[0].set_title('Autocorrelation (ACF) - Hourly Data', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Lag (hours)')
# Mark the key periodicities
for lag, label in [(24, '24h'), (48, '48h'), (168, '1 week')]:
    if lag <= 168:
        axes[0].axvline(lag, color='red', linestyle=':', alpha=0.5)
        axes[0].text(lag, axes[0].get_ylim()[1]*0.9, label, ha='center', fontsize=9, color='red')

# PACF for the first 48 lags
plot_pacf(hourly_gap, lags=48, ax=axes[1], alpha=0.05, method='ywm')
axes[1].set_title('Partial Autocorrelation (PACF) - Hourly Data', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Lag (hours)')

plt.tight_layout()
plt.show()

print("\nInterpretation for the thesis:")
print("  - Peaks at lags 24 and 168 confirm daily and weekly periodicity")
print("  - This justifies using 24h or 168h windows for the models")
print("  - LSTMs and Autoencoders can capture these temporal dependencies")

## 9. Preparation for the Anomaly Detection Models

Here we define the time windows and the normalisation used by the models.

In [ ]:
# Features used by the models
FEATURE_COLS = [
    'Global_active_power', 'Global_reactive_power',
    'Voltage', 'Global_intensity',
    'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]
# Keep only the columns that exist
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_clean.columns]

print(f"Features for models: {FEATURE_COLS}")
print(f"Number of features: {len(FEATURE_COLS)}")

# Summary of the clean dataset ready for modelling
print(f"\nClean dataset ready for modelling:")
print(f"  Train: {df_train.shape[0]:,} rows (before cleaning)")
print(f"  Test:  {df_test.shape[0]:,} rows (before cleaning)")
print(f"  Features: {len(FEATURE_COLS)}")
print(f"  Resolution: 1 minute")

In [ ]:
# Min-Max normalisation (keeps the relative scale across features)
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Fit on the training data ONLY, to avoid data leakage
train_features = df_train[FEATURE_COLS].dropna() if all(c in df_train.columns for c in FEATURE_COLS) else df_clean[FEATURE_COLS].iloc[:int(len(df_clean)*0.8)]
scaler.fit(train_features)

# Report the per-feature ranges
print("Per-feature ranges (before normalising):")
for i, col in enumerate(FEATURE_COLS):
    print(f"  {col}: [{scaler.data_min_[i]:.3f}, {scaler.data_max_[i]:.3f}]")

In [ ]:
# Build sliding time windows.
# This is the input format for the autoencoders and LSTMs.

def make_windows(data, window_size=60, step=1):
    """
    Build fixed-size sliding windows.

    Args:
        data: numpy array of shape (n_samples, n_features)
        window_size: window length in minutes
        step: stride between windows (1 = every possible window)

    Returns:
        array of shape (n_windows, window_size, n_features)
    """
    windows = []
    for i in range(0, len(data) - window_size, step):
        windows.append(data[i:i + window_size])
    return np.array(windows)


# Quick demo with a 60-minute (1-hour) window
sample_data = scaler.transform(train_features.iloc[:10000])
windows = make_windows(sample_data, window_size=60, step=60)

print(f"Window construction demo:")
print(f"  Input data: {sample_data.shape}")
print(f"  Generated windows: {windows.shape}")
print(f"  -> {windows.shape[0]} windows of {windows.shape[1]} minutes x {windows.shape[2]} features")

## 10. Summary and Next Steps

### Key findings from this EDA:

1. **Clear daily pattern**: consumption bottoms out in the early morning (~3-5h) and peaks at midday and at night
2. **Weekday/weekend contrast**: the profile shifts, with more daytime consumption at weekends
3. **Seasonality**: consumption is higher in winter (heating and electric water heater)
4. **Strong autocorrelation**: marked periodicities at 24h and 168h (weekly)
5. **~1.25% missing values**: low enough to handle with interpolation
6. **Natural anomalies**: a small fraction of points exceed |z| > 3 (normalised by hour)

### Following notebooks:

- **Notebook 02**: Injection of synthetic FDI attacks (scaling, offset, random noise)
- **Notebook 03**: Model 1 - Isolation Forest (baseline)
- **Notebook 04**: Model 2 - Dense Autoencoder
- **Notebook 05**: Model 3 - LSTM-Autoencoder
- **Notebook 06**: Model comparison and Edge analysis

In [ ]:
# Save the clean dataset for the following notebooks
OUTPUT_DIR = '/Volumes/Extreme Pro Particion 1TB/TFG/UCIrvine/'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_clean.to_csv(f'{OUTPUT_DIR}uci_clean_full.csv')
print(f"Clean dataset saved to {OUTPUT_DIR}uci_clean_full.csv")
print(f"  Rows: {df_clean.shape[0]:,}")
print(f"  Columns: {list(df_clean.columns)}")